### What you should record for Professor Rahman

Don't just report that the code ran. Keep notes on:

- whether the reported RQ1 metrics are reproduced;
- setup/dependency problems;
- differences between your result and the paper's result;
- which tests change prediction after dead-code perturbation;
- examples where the model appears sensitive to irrelevant code changes;
- anything surprising in the attribution results if you run RQ3.

Those observations are more valuable for the research exercise than simply saying "the artifact worked."


In [1]:
from pathlib import Path
import shutil

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "flakylens"

print("Kaggle inputs:")
for p in INPUT.iterdir():
    print(" ", p)

Kaggle inputs:
  /kaggle/input/datasets


In [3]:
# Find the extracted FlakyLens artifact
artifact_candidates = []

for p in INPUT.rglob("Testing_per_project.py"):
    if p.parent.name == "src":
        artifact_candidates.append(p.parent.parent)

if not artifact_candidates:
    raise FileNotFoundError(
        "FlakyLens artifact not found in /kaggle/input."
    )

ARTIFACT = artifact_candidates[0]

print("Found artifact:")
print(ARTIFACT)

Found artifact:
/kaggle/input/datasets/zaimataheri1704054/artifact/Understanding_and_Improving_FlakyTest_Classifiers_Artifact


In [4]:
# Copy artifact into working directory

if REPO.exists():
    shutil.rmtree(REPO)

shutil.copytree(ARTIFACT, REPO)

print("Artifact copied to:")
print(REPO)

Artifact copied to:
/kaggle/working/flakylens


In [5]:
# Verify artifact

DATASET = (
    REPO
    / "dataset"
    / "FlakyLens"
    / "FlakyLens_dataset_with_nonflaky_indented.csv"
)

TESTING = REPO / "src" / "Testing_per_project.py"
MODEL_README = REPO / "models" / "README.md"

print("README:", (REPO / "README.md").exists())
print("Dataset:", DATASET.exists())
print("Testing script:", TESTING.exists())
print("Model README:", MODEL_README.exists())

README: True
Dataset: True
Testing script: True
Model README: True


In [6]:
# Find uploaded model weights

weights = list(INPUT.rglob("*.pt")) + list(INPUT.rglob("*.pth"))

print("Model weights found:")

for p in weights:
    print(
        f"  {p} "
        f"({p.stat().st_size / 1024 / 1024:.1f} MB)"
    )

group1 = [
    p for p in weights
    if "project_group_1" in p.name.lower()
]

if not group1:
    raise FileNotFoundError(
        "project_group_1.pt was not found."
    )

SOURCE_WEIGHT = group1[0]

print("\nUsing:")
print(SOURCE_WEIGHT)

Model weights found:
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_3.pt (477.1 MB)
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_1.pt (477.1 MB)
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_4.pt (477.1 MB)
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_2.pt (477.1 MB)

Using:
/kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_1.pt


In [7]:
# Put the checkpoint inside the artifact's models directory

MODELS = REPO / "models"
MODELS.mkdir(exist_ok=True)

TARGET_WEIGHT = MODELS / SOURCE_WEIGHT.name

shutil.copy2(
    SOURCE_WEIGHT,
    TARGET_WEIGHT
)

print("Checkpoint copied to:")
print(TARGET_WEIGHT)

Checkpoint copied to:
/kaggle/working/flakylens/models/per_project_model_weights_on__dataset_project_group_1.pt


In [8]:
# Inspect exactly how the artifact loads the model

source_text = TESTING.read_text(errors="ignore")

for i, line in enumerate(source_text.splitlines(), 1):
    if (
        "model_weights_path" in line
        or "load_state_dict" in line
    ):
        print(f"{i}: {line}")

374: def run_experiment(dataset_path, model_weights_path, calculate_attribution, data_name, technique, perturbation):
527:             #print(model_weights_path+str(project_group)+'_with_noisy_train_data.pt') 
529:             model.load_state_dict(torch.load(model_weights_path+str(project_group)+'.pt'))'''
531:             #model.load_state_dict(torch.load(model_weights_path+str(project_group)+'_With_noisy_train_data.pt'))
532:             #model.load_state_dict(torch.load(model_weights_path+str(project_group)+'_Only_noisy_train_1024_data.pt'))
534:             #print(model_weights_path+'_project_group_'+str(project_group)+'.pt')
535:             model.load_state_dict(torch.load(model_weights_path+'_project_group_'+str(project_group)+'.pt'))
674:     model_weights_path = sys.argv[2] #"../results_per_project/So-Far-Good-Models/per_project_model_weights_on__dataset" #sys.argv[2]
682:     run_experiment(dataset_path, model_weights_path, calculate_attribution, data_name_dir, technique, pe

In [9]:
# Find RQ1 script

rq1_files = list(REPO.rglob("rq1.sh"))

if not rq1_files:
    print("No rq1.sh found.")
else:
    for rq1 in rq1_files:
        print("=" * 80)
        print(rq1.relative_to(REPO))
        print("=" * 80)
        print(rq1.read_text(errors="ignore"))

src/rq1.sh
#!/bin/bash
currentDir=$(pwd)
bash per_project_prediction.sh FlakyLens "BERT"
cd ../results/scripts
bash per_category_parse_result.sh ../per_Category_Evaluation_BERT-FlakyLens.txt
rm ../per_Category_Evaluation_BERT-FlakyLens.txt
cd $currentDir



In [11]:
# Inspect requirements, but DON'T install them yet

requirements = REPO / "requirements.txt"

if requirements.exists():
    print(requirements.read_text(errors="ignore"))
else:
    print("requirements.txt not found.")

beautifulsoup4==4.13.4
captum==0.7.0
huggingface_hub==0.26.2
imbalanced_learn==0.11.0
imblearn==0.0
interpret==0.6.1
interpret_core==0.6.1
ipython==8.12.3
javalang==0.13.0
matplotlib==3.6.3
numpy==1.23.5
openai==0.28.0
pandas==1.3.3
scikit_learn==1.3.2
scipy==1.9.1
seaborn==0.13.2
selenium==4.27.1
shap==0.41.0
torch>=2.3.0,<2.4.0
transformers==4.40.1
xgboost==1.7.5
tensorboard==2.13.0



In [12]:
!pip install -q captum==0.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.3 MB/s eta 0:00:00


In [13]:
# Check Kaggle's existing PyTorch/GPU environment

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [14]:
# Test whether the uploaded checkpoint can be read

checkpoint = torch.load(
    TARGET_WEIGHT,
    map_location="cpu"
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Number of entries:", len(checkpoint))
    print("First keys:")
    print(list(checkpoint.keys())[:20])

Checkpoint type: <class 'collections.OrderedDict'>
Number of entries: 203
First keys:
['bert.embeddings.word_embeddings.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.token_type_embeddings.weight', 'bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.output.dense.weight', 'bert.encoder.layer.0.attention.output.dense.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.0.output.dense.bias', '

In [15]:
from pathlib import Path

SRC = Path("/kaggle/working/flakylens/src")

# Fix the old Transformers AdamW import
for filename in ["Testing_per_project_group1.py", "Testing_per_project.py"]:
    path = SRC / filename
    
    if path.exists():
        text = path.read_text()
        
        old = "from transformers import AdamW, AutoTokenizer, AutoModel, AutoConfig"
        new = "from torch.optim import AdamW\nfrom transformers import AutoTokenizer, AutoModel, AutoConfig"
        
        if old in text:
            text = text.replace(old, new)
            path.write_text(text)
            print(f"✓ Fixed AdamW import in {filename}")
        else:
            print(f"✓ No old AdamW import found in {filename}")

✓ Fixed AdamW import in Testing_per_project.py


In [16]:
from pathlib import Path

SRC = Path("/kaggle/working/flakylens/src")

utils_file = SRC / "utils.py"

text = utils_file.read_text()

old = (
    "from transformers import AdamW, AutoTokenizer, AutoModel, AutoConfig, "
    "T5Tokenizer, T5ForConditionalGeneration, T5EncoderModel, "
    "RobertaTokenizer, AutoModelForSequenceClassification"
)

new = (
    "from torch.optim import AdamW\n"
    "from transformers import AutoTokenizer, AutoModel, AutoConfig, "
    "T5Tokenizer, T5ForConditionalGeneration, T5EncoderModel, "
    "RobertaTokenizer, AutoModelForSequenceClassification"
)

if old in text:
    text = text.replace(old, new, 1)
    utils_file.write_text(text)
    print("✓ Fixed AdamW import in utils.py")
else:
    print("⚠ Exact import line was not found.")
    print("\nFirst 15 lines of utils.py:\n")
    print("\n".join(text.splitlines()[:15]))

✓ Fixed AdamW import in utils.py


In [18]:
!pip install -q javalang==0.13.0

In [19]:
!pip install -q interpret==0.6.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 75.7 MB/s eta 0:00:00:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 48.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 19.1 MB/s eta 0:00:00


In [20]:
!pip install -q selenium==4.27.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 93.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 29.3 MB/s eta 0:00:00


In [21]:
from pathlib import Path

DATA_PROCESSING = Path(
    "/kaggle/working/flakylens/src/data_processing.py"
)

lines = DATA_PROCESSING.read_text().splitlines()

print("====================================================")
print("data_processing.py — tokenize_data()")
print("====================================================")

for i in range(425, 450):
    if i < len(lines):
        print(f"{i+1}: {lines[i]}")

data_processing.py — tokenize_data()
426:     # 2. Apply SMOTE
427:     smote = SMOTE(sampling_strategy='minority', random_state=49)
428:     x_train, y_train = smote.fit_resample(x_train, y_train)
429: 
430:     # 3. Convert the sparse matrix to dense dataframe (if necessary)
431:     x_train_df = pd.DataFrame(x_train.toarray(), columns=vectorizer.get_feature_names())
432:     x_train_series = x_train_df.apply(lambda row: ' '.join(row.map(str)), axis=1)
433:     return x_train_series, y_train
434: 
435: # convert code into tokens and then vector representation
436: class Tokenizer:
437:     def __init__(self, tokenizer, max_length=512):
438:         self.tokenizer = tokenizer
439:         self.max_length = max_length
440:     
441:     def tokenize_data(self, texts):
442:         tokens = self.tokenizer.batch_encode_plus(
443:             texts.tolist(),
444:             max_length=self.max_length,
445:             pad_to_max_length=True,
446:             truncation=True)
447:        

In [22]:
from pathlib import Path

DATA_PROCESSING = Path(
    "/kaggle/working/flakylens/src/data_processing.py"
)

text = DATA_PROCESSING.read_text()

old = """        tokens = self.tokenizer.batch_encode_plus(
            texts.tolist(),
            max_length=self.max_length,
            pad_to_max_length=True,
            truncation=True)"""

new = """        tokens = self.tokenizer(
            texts.tolist(),
            max_length=self.max_length,
            padding="max_length",
            truncation=True)"""

if old not in text:
    print("⚠ Original tokenizer code was not found.")
    
    # Show the actual section for inspection
    lines = text.splitlines()
    for i in range(435, 450):
        if i < len(lines):
            print(f"{i+1}: {lines[i]}")
else:
    text = text.replace(old, new, 1)
    DATA_PROCESSING.write_text(text)
    
    print("✓ Tokenizer compatibility patch applied.")
    print()
    print("Changed:")
    print("  batch_encode_plus(...)")
    print("        ↓")
    print("  tokenizer(...)")
    print()
    print("Padding behavior preserved:")
    print("  pad_to_max_length=True")
    print("        ↓")
    print('  padding="max_length"')

✓ Tokenizer compatibility patch applied.

Changed:
  batch_encode_plus(...)
        ↓
  tokenizer(...)

Padding behavior preserved:
  pad_to_max_length=True
        ↓
  padding="max_length"


In [23]:
from pathlib import Path

DATA_PROCESSING = Path(
    "/kaggle/working/flakylens/src/data_processing.py"
)

lines = DATA_PROCESSING.read_text().splitlines()

print("====================================================")
print("UPDATED tokenize_data()")
print("====================================================")

for i in range(435, 450):
    if i < len(lines):
        print(f"{i+1}: {lines[i]}")

UPDATED tokenize_data()
436: class Tokenizer:
437:     def __init__(self, tokenizer, max_length=512):
438:         self.tokenizer = tokenizer
439:         self.max_length = max_length
440:     
441:     def tokenize_data(self, texts):
442:         tokens = self.tokenizer(
443:             texts.tolist(),
444:             max_length=self.max_length,
445:             padding="max_length",
446:             truncation=True)
447:         return tokens
448: 
449:     def tokenize_training_data(self, train_text):
450:         return self.tokenize_data(train_text)


In [24]:
import torch
import transformers
import sys

print("====================================================")
print("ENVIRONMENT CHECK")
print("====================================================")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
else:
    print("⚠ PyTorch is running without CUDA/GPU.")

print("====================================================")

ENVIRONMENT CHECK
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
Transformers: 5.0.0

GPU available: True
GPU: Tesla T4
CUDA: 12.8
GPU count: 2


In [26]:
from pathlib import Path

script = Path(
    "/kaggle/working/flakylens/src/Testing_per_project.py"
)



text = script.read_text().splitlines()

print("====================================================")
print("DEVICE / FOLD CODE")
print("====================================================")

for i, line in enumerate(text, 1):
    if "device" in line.lower() or "NOW IN FOLD" in line:
        print(f"{i}: {line}")

DEVICE / FOLD CODE
161: def give_test_data_in_chunks(x_test_nparray, tokenizer, model, batch_size, device, project_group, label, Y_test, dataset_category, attributions_dir, calculate_attribution=False):
195:         test_seq = torch.tensor(tokens_test['input_ids']).to(device).long()
196:         test_mask = torch.tensor(tokens_test['attention_mask']).to(device).long()
376:     device, ml_technique, dataset_category, output_layer, where_data_comes = init_setup(technique, data_name)
420:             print(" NOW IN FOLD NUMBER", project_group)
538:             model.to(device)
543:                 preds, html_content, attribution_csvfile_name, confidence_scores = give_test_data_in_chunks(X_test, tokenizer, model, batch_size, device, project_group, label, y_test, dataset_category, attributions_dir, calculate_attribution)


In [29]:
from pathlib import Path

SRC = Path("/kaggle/working/flakylens/src")

# =========================================================
# 1. Fix AdamW import in Testing_per_project.py
# =========================================================

TESTING = SRC / "Testing_per_project.py"

text = TESTING.read_text()

old = "from transformers import AdamW, AutoTokenizer, AutoModel, AutoConfig"
new = (
    "from torch.optim import AdamW\n"
    "from transformers import AutoTokenizer, AutoModel, AutoConfig"
)

if old in text:
    text = text.replace(old, new, 1)
    print("✓ AdamW import fixed in Testing_per_project.py")
else:
    print("✓ AdamW import already compatible.")

# =========================================================
# 2. Fix batch_encode_plus() in Testing_per_project.py
# =========================================================

old_tokenizer = (
    "tokens_test = tokenizer.batch_encode_plus("
    "test_data, max_length=max_length, "
    "pad_to_max_length=True, truncation=True)"
)

new_tokenizer = (
    'tokens_test = tokenizer('
    'test_data, max_length=max_length, '
    'padding="max_length", truncation=True)'
)

if old_tokenizer in text:
    text = text.replace(
        old_tokenizer,
        new_tokenizer,
        1
    )
    print("✓ Test tokenizer compatibility fixed.")
else:
    print("✓ Test tokenizer already compatible.")

TESTING.write_text(text)

# =========================================================
# 3. Fix AdamW import in utils.py
# =========================================================

UTILS = SRC / "utils.py"

text = UTILS.read_text()

old_utils = (
    "from transformers import AdamW, AutoTokenizer, AutoModel, "
    "AutoConfig, T5Tokenizer, T5ForConditionalGeneration, "
    "T5EncoderModel, RobertaTokenizer, "
    "AutoModelForSequenceClassification"
)

new_utils = (
    "from torch.optim import AdamW\n"
    "from transformers import AutoTokenizer, AutoModel, "
    "AutoConfig, T5Tokenizer, T5ForConditionalGeneration, "
    "T5EncoderModel, RobertaTokenizer, "
    "AutoModelForSequenceClassification"
)

if old_utils in text:
    text = text.replace(old_utils, new_utils, 1)
    print("✓ AdamW import fixed in utils.py")
else:
    print("✓ utils.py AdamW import already compatible.")

UTILS.write_text(text)

# =========================================================
# 4. Fix tokenizer in data_processing.py
# =========================================================

DATA_PROCESSING = SRC / "data_processing.py"

text = DATA_PROCESSING.read_text()

old_dp = """        tokens = self.tokenizer.batch_encode_plus(
            texts.tolist(),
            max_length=self.max_length,
            pad_to_max_length=True,
            truncation=True)"""

new_dp = """        tokens = self.tokenizer(
            texts.tolist(),
            max_length=self.max_length,
            padding="max_length",
            truncation=True)"""

if old_dp in text:
    text = text.replace(old_dp, new_dp, 1)
    print("✓ Training tokenizer compatibility fixed.")
else:
    print("✓ data_processing.py tokenizer already compatible.")

DATA_PROCESSING.write_text(text)

print("\n====================================================")
print("✓ ORIGINAL TESTING_PER_PROJECT.PY IS READY")
print("====================================================")

✓ AdamW import already compatible.
✓ Test tokenizer compatibility fixed.
✓ utils.py AdamW import already compatible.
✓ data_processing.py tokenizer already compatible.

✓ ORIGINAL TESTING_PER_PROJECT.PY IS READY


In [30]:
from pathlib import Path
import subprocess
import sys

# =========================================================
# CONFIGURATION
# =========================================================

REPO = Path("/kaggle/working/flakylens")
SRC = REPO / "src"

DATASET = (
    REPO
    / "dataset"
    / "FlakyLens"
    / "FlakyLens_dataset_with_nonflaky_indented.csv"
)

INPUT_ROOT = Path("/kaggle/input")

# =========================================================
# 1. Find all four model weights
# =========================================================

print("====================================================")
print("SEARCHING FOR FLAKYLENS MODEL WEIGHTS")
print("====================================================")

model_files = {}

for group in range(1, 5):

    filename = (
        f"per_project_model_weights_on__dataset"
        f"_project_group_{group}.pt"
    )

    matches = list(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"\n❌ Group {group} weight was not found.\n"
            f"Expected filename:\n{filename}\n\n"
            f"Check that the model_weights dataset is attached "
            f"to this Kaggle notebook."
        )

    if len(matches) > 1:
        print(f"⚠ Multiple files found for Group {group}:")
        for m in matches:
            print("   ", m)
        print("Using:", matches[0])

    model_files[group] = matches[0]

    print(f"✓ Group {group}:")
    print(f"  {model_files[group]}")

print("\n✓ ALL FOUR MODEL WEIGHTS FOUND.")

# =========================================================
# 2. Verify that all four weights are in the SAME folder
# =========================================================

model_dirs = {
    path.parent
    for path in model_files.values()
}

print("\n====================================================")
print("MODEL DIRECTORY CHECK")
print("====================================================")

for d in model_dirs:
    print(d)

if len(model_dirs) != 1:
    raise RuntimeError(
        "\n❌ The four model weights are not in the same folder."
    )

MODEL_DIR = next(iter(model_dirs))

# =========================================================
# 3. Construct the model PREFIX
# =========================================================
#
# The artifact itself adds:
#
# _project_group_1.pt
# _project_group_2.pt
# _project_group_3.pt
# _project_group_4.pt
#
# Therefore we provide only:
#
# per_project_model_weights_on__dataset
#
# =========================================================

MODEL_PREFIX = (
    MODEL_DIR /
    "per_project_model_weights_on__dataset"
)

print("\n====================================================")
print("MODEL PREFIX")
print("====================================================")
print(MODEL_PREFIX)

# Verify exactly what the artifact will look for
for group in range(1, 5):

    expected = Path(
        str(MODEL_PREFIX)
        + f"_project_group_{group}.pt"
    )

    if not expected.exists():
        raise FileNotFoundError(
            f"Expected model does not exist:\n{expected}"
        )

    print(f"✓ Group {group} verified")

# =========================================================
# 4. Verify FlakyLens dataset
# =========================================================

if not DATASET.exists():
    raise FileNotFoundError(
        f"\n❌ Dataset not found:\n{DATASET}"
    )

print("\n✓ FlakyLens dataset found:")
print(DATASET)

# =========================================================
# 5. Create a temporary ALL-GROUPS execution script
# =========================================================

original_script = SRC / "Testing_per_project.py"
allgroups_script = SRC / "Testing_per_project.py"

if not original_script.exists():
    raise FileNotFoundError(
        f"Testing_per_project.py not found:\n{original_script}"
    )

text = original_script.read_text()

# ---------------------------------------------------------
# Keep the original stopping condition:
#
# if project_group == 5:
#
# This means groups 1, 2, 3 and 4 are processed.
# ---------------------------------------------------------

if "if project_group == 5:" not in text:
    raise RuntimeError(
        "Could not find the expected project-group "
        "stopping condition."
    )

# =========================================================
# 6. Apply the AdamW compatibility fix to the temporary
#    execution script
# =========================================================

old_import = (
    "from transformers import AdamW, AutoTokenizer, "
    "AutoModel, AutoConfig"
)

new_import = (
    "from torch.optim import AdamW\n"
    "from transformers import AutoTokenizer, "
    "AutoModel, AutoConfig"
)

if old_import in text:
    text = text.replace(
        old_import,
        new_import,
        1
    )
    print("\n✓ Applied AdamW compatibility fix.")
else:
    print("\n✓ No old AdamW import found in main script.")

# Save temporary script
allgroups_script.write_text(text)

print("\n✓ All-groups script created:")
print(allgroups_script)

# =========================================================
# 7. Run ALL FOUR project groups
# =========================================================

cmd = [
    sys.executable,
    "-W", "ignore",

    "Testing_per_project.py",

    # Dataset
    "../dataset/FlakyLens/"
    "FlakyLens_dataset_with_nonflaky_indented.csv",

    # IMPORTANT:
    # Model PREFIX, not a .pt filename
    str(MODEL_PREFIX),

    # Normal prediction
    "calculate_attribution_False",

    # Data directory
    "FlakyLens_Categorization_PerProject-Data",

    # Technique
    "BERT-FlakyLens"
]

print("\n====================================================")
print("RUNNING FLAKYLENS — ALL FOUR PROJECT GROUPS")
print("====================================================")

print("\nModels:")
for group in range(1, 5):
    print(
        f"  Group {group}: "
        f"{model_files[group].name}"
    )

print("\nCommand:")
print(" ".join(cmd))

print("\n====================================================")
print("STARTING EXPERIMENT")
print("====================================================\n")

result = subprocess.run(
    cmd,
    cwd=SRC,
    text=True
)

# =========================================================
# 8. Final status
# =========================================================

print("\n====================================================")
print("PROCESS FINISHED")
print("====================================================")

print("Return code:", result.returncode)

if result.returncode == 0:
    print("\n✓ ALL FOUR PROJECT GROUPS COMPLETED SUCCESSFULLY.")
else:
    print("\n✗ FLAKYLENS EXECUTION FAILED.")
    print("Check the traceback above.")

SEARCHING FOR FLAKYLENS MODEL WEIGHTS
✓ Group 1:
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_1.pt
✓ Group 2:
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_2.pt
✓ Group 3:
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_3.pt
✓ Group 4:
  /kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset_project_group_4.pt

✓ ALL FOUR MODEL WEIGHTS FOUND.

MODEL DIRECTORY CHECK
/kaggle/input/datasets/zaimataheri1704054/modelsw

MODEL PREFIX
/kaggle/input/datasets/zaimataheri1704054/modelsw/per_project_model_weights_on__dataset
✓ Group 1 verified
✓ Group 2 verified
✓ Group 3 verified
✓ Group 4 verified

✓ FlakyLens dataset found:
/kaggle/working/flakylens/dataset/FlakyLens/FlakyLens_dataset_with_nonflaky_indented.csv

✓ No old AdamW import found in main script.

✓ All-groups script created:
/kaggl

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2748.49it/s, Materializing param=pooler.dense.weight]                               


 NOW IN FOLD NUMBER 1
 NOW IN FOLD NUMBER 2
 NOW IN FOLD NUMBER 3
 NOW IN FOLD NUMBER 4

PROCESS FINISHED
Return code: 0

✓ ALL FOUR PROJECT GROUPS COMPLETED SUCCESSFULLY.


In [32]:
from pathlib import Path

REPO = Path("/kaggle/working/flakylens")
RESULTS = REPO / "results"
SRC = REPO / "src"

print("====================================================")
print("CHECKING FLAKYLENS RQ1 RESULTS")
print("====================================================")

# Search for the evaluation result generated by Testing_per_project.py
result_files = list(REPO.rglob("*per_Category_Evaluation*"))

if result_files:
    print("\n✓ Evaluation result files found:\n")
    for f in result_files:
        print(f)
else:
    print("\n⚠ No per_Category_Evaluation file found.")

# Show result directory contents
print("\n====================================================")
print("RESULTS DIRECTORY")
print("====================================================")

if RESULTS.exists():
    for f in sorted(RESULTS.rglob("*")):
        if f.is_file():
            print(f)
else:
    print("Results directory not found.")

CHECKING FLAKYLENS RQ1 RESULTS

✓ Evaluation result files found:

/kaggle/working/flakylens/results/per_Category_Evaluation_BERT-FlakyLens.txt
/kaggle/working/flakylens/results/per_Category_Evaluation_BERT.txt

RESULTS DIRECTORY
/kaggle/working/flakylens/results/FlakyLens_Result_Found_By_Author.csv
/kaggle/working/flakylens/results/per_Category_Evaluation_BERT-FlakyLens.txt
/kaggle/working/flakylens/results/per_Category_Evaluation_BERT.txt
/kaggle/working/flakylens/results/qwen_Result_Found_By_Author.csv
/kaggle/working/flakylens/results/scripts/parse_result.sh
/kaggle/working/flakylens/results/scripts/per_category_parse_result.py
/kaggle/working/flakylens/results/scripts/per_category_parse_result.sh
/kaggle/working/flakylens/results/scripts/result_to_show_in_paper_for_imbalanced_data.py
/kaggle/working/flakylens/results/weighted_avg_for_cv_BERT-FlakyLens.txt


In [33]:
from pathlib import Path
import subprocess

REPO = Path("/kaggle/working/flakylens")
RESULTS = REPO / "results"
SCRIPTS = RESULTS / "scripts"

RAW_RESULT = RESULTS / "per_Category_Evaluation_BERT-FlakyLens.txt"
PARSER = SCRIPTS / "per_category_parse_result.sh"

print("====================================================")
print("PARSING FLAKYLENS RQ1 RESULTS")
print("====================================================")

if not RAW_RESULT.exists():
    raise FileNotFoundError(
        f"Raw RQ1 result not found:\n{RAW_RESULT}"
    )

if not PARSER.exists():
    raise FileNotFoundError(
        f"Parser not found:\n{PARSER}"
    )

print("✓ Raw result found:")
print(RAW_RESULT)

print("\n✓ Parser found:")
print(PARSER)

# ---------------------------------------------------------
# Run the original parser
# ---------------------------------------------------------

result = subprocess.run(
    ["bash", str(PARSER), str(RAW_RESULT)],
    cwd=SCRIPTS,
    text=True
)

print("\n====================================================")
print("PARSER FINISHED")
print("Return code:", result.returncode)
print("====================================================")

if result.returncode == 0:
    print("✓ RQ1 results parsed successfully.")
else:
    print("✗ Parser failed.")

PARSING FLAKYLENS RQ1 RESULTS
✓ Raw result found:
/kaggle/working/flakylens/results/per_Category_Evaluation_BERT-FlakyLens.txt

✓ Parser found:
/kaggle/working/flakylens/results/scripts/per_category_parse_result.sh
precision,recall,f1(Async),precision,recall,f1(Conc),precision,recall,f1(Time),precision,recall,f1(UC),precision,recall,f1(OD),precision,recall,f1(Non-flaky)
0.8380263157894736,0.9052631578947368,0.8681578947368421,0.8505405405405405,0.8097297297297298,0.8278378378378379,0.8636363636363636,0.9709090909090908,0.8930303030303031,0.8926829268292683,0.9292682926829267,0.9053658536585365,0.9716129032258063,0.8093548387096774,0.8534408602150536,1.0,1.0,1.0
macro_f1= 0.8995833333333333

PARSER FINISHED
Return code: 0
✓ RQ1 results parsed successfully.


/kaggle/working/flakylens/results/scripts/result_to_show_in_paper_for_imbalanced_data.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_weighted_f1 = df_formatted.groupby("Fold").apply(lambda x: x["Weighted F1"].sum() / x["Support"].sum())


In [34]:
from pathlib import Path

RESULT = Path(
    "/kaggle/working/flakylens/results/"
    "per_Category_Evaluation_BERT-FlakyLens.txt"
)

print("====================================================")
print("RAW FLAKYLENS RESULT")
print("====================================================")

text = RESULT.read_text()

print(text)

print("\n====================================================")
print("NUMBER OF LINES")
print("====================================================")

print(len(text.splitlines()))

RAW FLAKYLENS RESULT
1:0:[0.76, 0.91, 0.83, 35.0]
1:1:[0.84, 0.76, 0.8, 21.0]
1:2:[0.5, 1.0, 0.67, 9.0]
1:3:[0.78, 0.88, 0.82, 16.0]
1:4:[1.0, 0.41, 0.58, 27.0]
1:5:[1.0, 1.0, 1.0, 2324.0]
2:0:[0.83, 0.91, 0.87, 11.0]
2:1:[0.67, 0.5, 0.57, 4.0]
2:2:[1.0, 1.0, 1.0, 4.0]
2:3:[1.0, 1.0, 1.0, 7.0]
2:4:[1.0, 1.0, 1.0, 14.0]
2:5:[1.0, 1.0, 1.0, 1867.0]
3:0:[0.92, 1.0, 0.96, 12.0]
3:1:[1.0, 1.0, 1.0, 7.0]
3:2:[1.0, 0.88, 0.93, 8.0]
3:3:[1.0, 0.86, 0.92, 7.0]
3:4:[0.88, 1.0, 0.93, 7.0]
3:5:[1.0, 1.0, 1.0, 2175.0]
4:0:[0.94, 0.83, 0.88, 18.0]
4:1:[0.83, 1.0, 0.91, 5.0]
4:2:[1.0, 1.0, 1.0, 12.0]
4:3:[0.92, 1.0, 0.96, 11.0]
4:4:[0.96, 0.96, 0.96, 45.0]
4:5:[1.0, 1.0, 1.0, 1928.0]


NUMBER OF LINES
24
